# Gymnasium agent — starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-arena/competition-baseline/blob/main/gymnasium/agent_baseline.ipynb)

A minimal, runnable **single-agent RL** baseline for any ml-arena Gymnasium
competition (LunarLander, CartPole, the Atari suite, the MuJoCo control tasks, ...).
It defines a random-action `Agent`, deploys it, and shows the leaderboard.

> Open in Colab, run the cells top to bottom. Edit **two** things: your API token
> and `COMPETITION_ID` (each competition page shows its id in the URL). The default
> is `43` — **LunarLander-v3**.

The agent is deliberately trivial — it plays *random* actions. Beating it is the
whole point: replace `choose_action` with a policy you have trained.

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk==0.3.0" numpy   # installs the `mlarena` package

import mlarena

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page -> API Keys)
COMPETITION_ID = 43

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Define your agent

The next cell writes **`agent.py`** to disk with `%%writefile`. We upload the whole file
(so its `import`s come along) — that is what the worker runs. Keep it **self-contained**:
it is executed on its own, so it may not import the environment or other notebook cells,
only public packages available in the runtime.

The worker imports the symbol **`Agent`** and drives it with the flex contract:
`Agent()` (no constructor args), then **`setup(observation_space, action_space)`** once
(both arrive dict-encoded — decode with `flexkit.spaces.decode_space`, as the template
does), then **`choose_action(...)`** each step, expecting an action from `action_space`.
Return `None` once the episode is over (`terminated` or `truncated`).

In [ ]:
%%writefile agent.py
"""
Gymnasium Agent Template (single-agent RL) — flex_v1 contract.

flexkit's gym loop drives your agent:
  1. Agent()                                   zero-arg constructor
  2. setup(observation_space, action_space)    once, before the first episode
     (the spaces arrive dict-encoded — decode with flexkit.spaces.decode_space)
  3. choose_action(observation, reward, terminated, truncated, info, action_mask)
     once per step; return the action, or None once the episode ends
"""


class Agent:
    def __init__(self):
        self.observation_space = None
        self.action_space = None

    def setup(self, observation_space, action_space):
        # flexkit is provided by the platform runtime (not needed to run this
        # notebook locally); import it here so `import agent` works anywhere.
        from flexkit.spaces import decode_space
        self.observation_space = decode_space(observation_space)
        self.action_space = decode_space(action_space)
        return True

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        # TODO: replace this random policy with your trained model.
        return self.action_space.sample()


## 2. (optional) Sanity-check it locally

The real Gymnasium spaces come from the environment at run time; here we just fake a
discrete action space to confirm `choose_action` returns a valid action.

In [ ]:
# Local smoke test — on the platform, flexkit calls setup() with the real specs.
# Here we set action_space directly so the notebook runs without flexkit.
from agent import Agent


class _FakeDiscrete:
    def __init__(self, n):
        self.n = n

    def sample(self):
        import random
        return random.randrange(self.n)


agent = Agent()
agent.action_space = _FakeDiscrete(4)   # setup() does this for you on the platform
print("sample action:", agent.choose_action(observation=[0.0] * 8))


## 3. Submit

This uploads `agent.py`, deploys it, and streams status until it scores.

In [ ]:
# Uploads agent.py (with its imports intact), then creates + deploys the attachment.
result = client.submit(competition_id=COMPETITION_ID, files=["agent.py"])
print(result)

# Stream status until the run reaches a terminal state (deploy -> queue -> run -> scored).
for line in client.tail_logs(COMPETITION_ID, result["attache_agent_id"]):
    print(line)

## 4. Leaderboard

In [ ]:
client.leaderboard(COMPETITION_ID)

## 5. Where to go from here

The random agent is just a launch pad. To climb the leaderboard:

- **Train a policy offline** (Stable-Baselines3 / CleanRL: PPO, DQN, SAC), save the
  weights, and load them in **`setup()`** (the per-agent init the platform calls with a
  startup deadline) — upload the weights file alongside `agent.py`
  with `client.submit(COMPETITION_ID, files=["agent.py", "model.zip"])`.
- **Match the runtime** to your framework with
  `client.submit(..., files=["agent.py"], runtime={"language": "python", "framework": "torch"})`.
- **Reuse this notebook** for any single-agent Gymnasium/Atari/MuJoCo competition —
  only `COMPETITION_ID` changes; the `Agent` contract is identical.